#Import

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import date

#Get Data From Bronze

In [0]:
# Load location data from Azure Data Lake Storage
df = spark.read.format("parquet")\
    .option("inferSchema",True)\
    .load("abfss://bronze@storenapatdatalake.dfs.core.windows.net/erp_LOC_A101")
df.limit(10).display()

CID,CNTRY
AW-00011000,Australia
AW-00011001,Australia
AW-00011002,Australia
AW-00011003,Australia
AW-00011004,Australia
AW-00011005,Australia
AW-00011006,Australia
AW-00011007,Australia
AW-00011008,Australia
AW-00011009,Australia


#Data Transformation

##Trimming

In [0]:
# Remove leading and trailing spaces from string columns
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,trim(col(field.name)))
df.limit(10).display()

CID,CNTRY
AW-00011000,Australia
AW-00011001,Australia
AW-00011002,Australia
AW-00011003,Australia
AW-00011004,Australia
AW-00011005,Australia
AW-00011006,Australia
AW-00011007,Australia
AW-00011008,Australia
AW-00011009,Australia


##Customer ID Cleanup

In [0]:
# Remove hyphens from customer IDs
df = df.withColumn("CID",regexp_replace(col("CID"),"-",""))
df.limit(10).display()

CID,CNTRY
AW00011000,Australia
AW00011001,Australia
AW00011002,Australia
AW00011003,Australia
AW00011004,Australia
AW00011005,Australia
AW00011006,Australia
AW00011007,Australia
AW00011008,Australia
AW00011009,Australia


##Country Normalization

In [0]:
df.select("CNTRY").distinct().display()

CNTRY
Germany
France
null
United States
DE
US
USA
Canada
""
Australia


In [0]:
# Standardize country codes and handle missing values
df = df.withColumn(
    "CNTRY",when(
        upper(col("CNTRY")) == "DE", "Germany"
    ).when(
        upper(col("CNTRY")).isin("US","USA"), "United States"
    ).when(
        (col("CNTRY").isNull()) | ((col("CNTRY") == "")) , "n/a"
    ).otherwise(col("CNTRY"))
)
df.select("CNTRY").distinct().display()

CNTRY
Germany
France
United States
n/a
Canada
Australia
United Kingdom


##Renaming Columns

In [0]:
# Rename source columns to business-friendly names
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}

for old_name , new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)
df.limit(10).display()


customer_number,country
AW00011000,Australia
AW00011001,Australia
AW00011002,Australia
AW00011003,Australia
AW00011004,Australia
AW00011005,Australia
AW00011006,Australia
AW00011007,Australia
AW00011008,Australia
AW00011009,Australia


##Create silver record_priority to filter out duplicate id

In [0]:
# Assign a row number to each customer ID
w = Window.partitionBy(df["customer_number"]).orderBy(lit(1))
df = df.withColumn("record_priority",row_number().over(w))
df.limit(10).display()

customer_number,country,record_priority
AW00011000,Australia,1
AW00011001,Australia,1
AW00011002,Australia,1
AW00011003,Australia,1
AW00011004,Australia,1
AW00011005,Australia,1
AW00011006,Australia,1
AW00011007,Australia,1
AW00011008,Australia,1
AW00011009,Australia,1


In [0]:
# Keep one record per customer ID
df = df.filter(col("record_priority") == 1)
df.limit(10).display()

customer_number,country,record_priority
AW00011000,Australia,1
AW00011001,Australia,1
AW00011002,Australia,1
AW00011003,Australia,1
AW00011004,Australia,1
AW00011005,Australia,1
AW00011006,Australia,1
AW00011007,Australia,1
AW00011008,Australia,1
AW00011009,Australia,1


#Data Writing

In [0]:
# Save cleaned location data in Parquet format
df.write.format("parquet").\
    mode("overwrite").\
    option("path","abfss://silver@storenapatdatalake.dfs.core.windows.net/erp_LOC_A101")\
    .save()

In [0]:
%sql
SELECT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/erp_LOC_A101` LIMIT 10

customer_number,country,record_priority
AW00011000,Australia,1
AW00011001,Australia,1
AW00011002,Australia,1
AW00011003,Australia,1
AW00011004,Australia,1
AW00011005,Australia,1
AW00011006,Australia,1
AW00011007,Australia,1
AW00011008,Australia,1
AW00011009,Australia,1


In [0]:
%sql
SELECT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/erp_LOC_A101` WHERE record_priority > 1 ORDER BY record_priority DESC LIMIT 10

customer_number,country,record_priority
